# KOSIS 통계표 구조와 뉴스 주장 검증 가능성 분석

목표: 통계표의 **표 이름·분류/항목·단위·시점**을 조사하고, 각 표가 검증할 수 있는 주장을 요약한다. 또한 뉴스 표현과 표 이름의 차이, 상대 시점 변환, KOSIS 통계 존재 비율(판단불가 예상치), API 활용법을 측정한다.

> API 키는 `.env`의 `KOSIS_API_KEY`를 사용하며 출력하지 않는다. API 호출 결과는 실행 시점에 따라 달라질 수 있다.

In [ ]:
import json, os, re, time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import pandas as pd

ROOT = Path.cwd()
CSV_PATH = ROOT / 'AI_기반_뉴스_사실검증_시스템_프로젝트_데이터_(1).csv'
ENV_PATH = ROOT / '.env'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

def load_env(path=ENV_PATH):
    for raw in path.read_text(encoding='utf-8-sig').splitlines():
        line = raw.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            value = value.strip().strip(chr(34)).strip(chr(39))
            os.environ.setdefault(key.strip(), value)

load_env()
API_KEY = os.getenv('KOSIS_API_KEY', '').strip()
assert API_KEY, '.env에 KOSIS_API_KEY가 필요합니다.'
print('환경 설정 완료 (API 키는 표시하지 않음)')

## 1. KOSIS API 함수

- `statisticsSearch.do`: 뉴스 지표 표현으로 후보 통계표 검색
- `statisticsData.do?method=getMeta`: 표명(`TBL`), 수록기간(`PRD`), 분류·항목·단위(`ITM`) 조회
- 실제 수치 조회는 후보 표와 분류/항목 코드를 확정한 뒤 `statisticsParameterData.do`를 사용한다.

In [ ]:
def kosis_get(url, **params):
    params.update(apiKey=API_KEY, format='json', jsonVD='Y')
    req = Request(f'{url}?{urlencode(params)}', headers={'User-Agent': 'kosis-claim-analysis/1.0'})
    with urlopen(req, timeout=40) as response:
        return json.loads(response.read().decode('utf-8'))

def search_tables(keyword, result_count=10):
    return kosis_get(
        'https://kosis.kr/openapi/statisticsSearch.do', method='getList',
        searchNm=keyword, sort='RANK', startCount=1, resultCount=result_count
    )

def table_meta(org_id, tbl_id, meta_type):
    return kosis_get(
        'https://kosis.kr/openapi/statisticsData.do', method='getMeta',
        type=meta_type, orgId=org_id, tblId=tbl_id
    )

def inspect_table(org_id, tbl_id):
    title = table_meta(org_id, tbl_id, 'TBL')
    periods = table_meta(org_id, tbl_id, 'PRD')
    items = table_meta(org_id, tbl_id, 'ITM')
    return {'ORG_ID': org_id, 'TBL_ID': tbl_id, 'title': title, 'periods': periods, 'items': items}

search_sample = search_tables('청년 실업률', 5)
pd.DataFrame(search_sample)[[c for c in ['TBL_NM','ORG_NM','STRT_PRD_DE','END_PRD_DE','TBL_ID'] if c in pd.DataFrame(search_sample).columns]]

## 2. 표 구조 조사 및 '검증 가능한 주장' 요약

아래 기본 표 목록은 앞 단계에서 확인한 주민등록인구 표다. 다른 분야를 조사하려면 `TARGET_TABLES`에 검색 결과의 기관코드와 통계표ID를 추가한다.

In [ ]:
TARGET_TABLES = [
    ('101', 'DT_1B040A3'),  # 행정구역·성별 인구수
    ('101', 'DT_1B040B3'),  # 행정구역별 주민등록세대수
]

def unique_values(rows, key, limit=12):
    values = list(dict.fromkeys(str(r.get(key, '')) for r in rows if r.get(key)))
    return values[:limit]

def summarize_claim_scope(meta):
    items = meta['items']
    title = meta['title'][0].get('TBL_NM', meta['TBL_ID']) if meta['title'] else meta['TBL_ID']
    dimensions = list(dict.fromkeys(r.get('OBJ_NM') for r in items if r.get('OBJ_NM')))
    measures = [r.get('ITM_NM') for r in items if r.get('OBJ_ID') == 'ITEM']
    units = unique_values(items, 'UNIT_NM')
    periods = [f"{p.get('PRD_SE')}: {p.get('STRT_PRD_DE')}~{p.get('END_PRD_DE')}" for p in meta['periods']]
    measure_text = ', '.join(measures[:6]) or '분류 항목별 값'
    dim_text = ', '.join(d for d in dimensions if d != '항목')
    claim = f"{dim_text or '대상'} 기준 {measure_text}의 규모·구성·시점 간 증감 주장"
    return {
        '통계표ID': meta['TBL_ID'], '통계표명': title, '분류': ', '.join(dimensions),
        '주요항목': measure_text, '단위': ', '.join(units), '수록시점': ' / '.join(periods),
        '검증 가능한 주장 요약': claim
    }

table_details = []
for org_id, tbl_id in TARGET_TABLES:
    table_details.append(inspect_table(org_id, tbl_id))
    time.sleep(0.15)
table_summary = pd.DataFrame([summarize_claim_scope(x) for x in table_details])
table_summary.to_csv(OUTPUT_DIR / 'kosis_table_claim_summary.csv', index=False, encoding='utf-8-sig')
table_summary

## 3. 뉴스 표현과 통계표 이름은 얼마나 다른가?

원문 전체를 통계표명과 바로 비교하면 보통 맞지 않는다. 숫자·시점·지역·대상 표현을 분리하고 핵심 지표 키워드(예: `청년+실업률`, `서울+인구`)로 검색한 뒤 후보 표를 재순위화해야 한다. 아래 코드는 검색어와 후보 표명의 문자 n-gram 유사도를 계산한다. 이는 베이스라인이며, 최종 시스템에서는 동의어 사전과 임베딩 검색을 추가한다.

In [ ]:
STOPWORDS = {'올해','지난해','작년','금년','전국','기준','기록','증가','감소','상승','하락'}
def normalize(text):
    return re.sub(r'[^0-9A-Za-z가-힣]', '', str(text)).lower()

def char_ngrams(text, n=2):
    text = normalize(text)
    return {text[i:i+n] for i in range(max(0, len(text)-n+1))}

def name_similarity(a, b):
    x, y = char_ngrams(a), char_ngrams(b)
    return len(x & y) / len(x | y) if x | y else 0.0

NEWS_EXPRESSIONS = ['청년 실업률', '서울 인구', '출생아 수', '소비자물가 상승률', '1인 가구 비중']
comparison_rows = []
for query in NEWS_EXPRESSIONS:
    candidates = search_tables(query, 5)
    for rank, row in enumerate(candidates, 1):
        comparison_rows.append({
            '뉴스 표현': query, '순위': rank, '통계표명': row.get('TBL_NM'),
            '유사도': round(name_similarity(query, row.get('TBL_NM', '')), 3),
            '통계표ID': row.get('TBL_ID'), '기관': row.get('ORG_NM')
        })
    time.sleep(0.15)
name_match_df = pd.DataFrame(comparison_rows)
name_match_df.groupby('뉴스 표현').head(3)

**해석 기준**

- 1위 표명이 뉴스 표현과 거의 같음: 이름 직접 매칭 가능
- 관련 후보는 있으나 표명이 다름: 중간 키워드·동의어·분류정보 필요
- 후보 없음/무관한 후보만 있음: KOSIS 미수록 또는 검색어 재작성 필요

검색 결과 존재는 검증 가능을 의미하지 않는다. 반드시 분류·항목·단위·수록기간이 주장의 조건과 일치하는지 확인해야 한다.

## 4. 시점·단위와 '올해·지난해' 정규화

KOSIS 수록주기는 `D/M/Q/S/Y/F/IR` 등이며, 실제 메타 응답은 `2026.06`, `2025`처럼 표시될 수 있다. 상대 시점은 **기사 발행일 기준**으로 변환해야 한다. 현재 날짜 기준으로 바꾸면 과거 기사를 잘못 검증하게 된다.

In [ ]:
def resolve_relative_time(text, published_at):
    published = pd.Timestamp(published_at)
    replacements = {
        '올해': str(published.year), '금년': str(published.year),
        '지난해': str(published.year - 1), '작년': str(published.year - 1),
        '재작년': str(published.year - 2),
        '지난달': (published - pd.DateOffset(months=1)).strftime('%Y.%m'),
    }
    normalized = str(text)
    for source, target in replacements.items():
        normalized = normalized.replace(source, target)
    return normalized

examples = pd.DataFrame({
    '기사문장': ['올해 인구가 감소했다', '지난해 실업률은 3%였다', '지난달 출생아 수가 증가했다'],
    '발행일': ['2025-08-10', '2024-03-02', '2026-01-15']
})
examples['정규화'] = examples.apply(lambda x: resolve_relative_time(x['기사문장'], x['발행일']), axis=1)
examples

단위는 단순 문자열 일치가 아니라 변환 규칙이 필요하다. 예: `만 명 → 명(×10,000)`, `억원 → 원(×100,000,000)`, `%`와 `%p`는 서로 다른 개념이다. 누적값·평균값·지수·증감률도 원자료 값과 직접 비교하면 안 된다.

In [ ]:
UNIT_MULTIPLIER = {'명': 1, '천명': 1_000, '만명': 10_000, '원': 1, '만원': 10_000, '억원': 100_000_000}
def convert_value(value, source_unit, target_unit):
    if source_unit in {'%', '%p'} or target_unit in {'%', '%p'}:
        if source_unit != target_unit:
            raise ValueError('%와 %p는 자동 변환하지 않습니다.')
        return value
    return value * UNIT_MULTIPLIER[source_unit] / UNIT_MULTIPLIER[target_unit]

convert_value(5.2, '만명', '명')

## 5. 뉴스 지표의 KOSIS 존재 비율과 '판단불가' 예상

정확한 비율을 얻으려면 기사에서 수치 주장과 지표명을 먼저 추출하고, 검색 후보를 사람이 라벨링해야 한다. `검색 결과가 1개 이상`은 낙관적 상한선일 뿐이다. 아래에서는 CSV 컬럼을 확인하고 텍스트 컬럼을 자동 선택한다. OneDrive 파일이 오프라인이면 먼저 파일을 로컬에 내려받아야 한다.

In [ ]:
def read_csv_flexible(path):
    errors = []
    for encoding in ['utf-8-sig', 'utf-8', 'cp949']:
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except Exception as exc:
            errors.append(f'{encoding}: {exc}')
    raise OSError('CSV를 읽지 못했습니다. OneDrive에서 로컬 다운로드 여부를 확인하세요.\n' + '\n'.join(errors))

news_df = read_csv_flexible(CSV_PATH)
print('shape:', news_df.shape)
display(pd.DataFrame({'column': news_df.columns, 'dtype': news_df.dtypes.astype(str)}))
news_df.head(3)

In [ ]:
# 컬럼 자동 선택 결과가 부정확하면 아래 두 값을 직접 수정하세요.
TEXT_HINTS = ['content', 'body', 'article', 'text', '본문', '내용']
DATE_HINTS = ['date', 'published', '작성일', '발행일', '일자']
def pick_column(columns, hints):
    lowered = {str(c).lower(): c for c in columns}
    return next((original for low, original in lowered.items() if any(h in low for h in hints)), None)

TEXT_COLUMN = pick_column(news_df.columns, TEXT_HINTS)
DATE_COLUMN = pick_column(news_df.columns, DATE_HINTS)
print('TEXT_COLUMN =', TEXT_COLUMN, '/ DATE_COLUMN =', DATE_COLUMN)
assert TEXT_COLUMN, '기사 본문 컬럼을 TEXT_COLUMN에 직접 지정하세요.'

In [ ]:
# PoC 베이스라인: 숫자가 포함된 문장에서 알려진 지표 어휘를 추출한다.
# 실제 주장 추출 모델/LLM 결과가 있다면 metric 컬럼을 가진 DataFrame으로 교체한다.
METRIC_TERMS = ['인구','출생아','사망자','가구','실업률','고용률','취업자','물가','소비자물가','GDP','성장률','소득','수출','수입']
NUMERIC_PATTERN = re.compile(r'\d[\d,.]*\s*(?:%p|%|명|가구|원|건|배|조|억|만|천)?')
claims = []
for idx, text in news_df[TEXT_COLUMN].dropna().astype(str).items():
    for sentence in re.split(r'(?<=[.!?다])\s+', text):
        if NUMERIC_PATTERN.search(sentence):
            terms = [term for term in METRIC_TERMS if term in sentence]
            for term in terms:
                claims.append({'row_id': idx, 'metric': term, 'claim_text': sentence[:500]})
claims_df = pd.DataFrame(claims).drop_duplicates(['row_id','metric'])
print('수치 주장 후보:', len(claims_df), '/ 고유 지표:', claims_df['metric'].nunique() if len(claims_df) else 0)
claims_df.head()

In [ ]:
# API 부담을 제한하기 위해 고유 지표만 조회한다. 프로젝트 범위에 맞춰 MAX_METRICS를 조정한다.
MAX_METRICS = 50
coverage_rows = []
for metric in claims_df['metric'].drop_duplicates().head(MAX_METRICS):
    results = search_tables(metric, 5)
    coverage_rows.append({
        'metric': metric, 'search_hit': bool(results), 'candidate_count': len(results),
        'top_table': results[0].get('TBL_NM') if results else None,
        'top_org_id': results[0].get('ORG_ID') if results else None,
        'top_tbl_id': results[0].get('TBL_ID') if results else None,
        'manual_label': ''  # MATCH / PARTIAL / NO_MATCH / NEED_REVIEW
    })
    time.sleep(0.12)
coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(OUTPUT_DIR / 'kosis_metric_coverage_review.csv', index=False, encoding='utf-8-sig')
search_hit_rate = coverage_df['search_hit'].mean() if len(coverage_df) else float('nan')
print(f'검색 후보 존재율(낙관적 상한): {search_hit_rate:.1%}')
coverage_df

`output/kosis_metric_coverage_review.csv`의 `manual_label`을 검토한 후 아래 셀을 실행한다. 운영상 검증 가능은 `MATCH`만, 또는 정책에 따라 `MATCH + PARTIAL`로 계산한다. `판단불가 예상률 = 1 - 검증 가능률`이다.

In [ ]:
review_path = OUTPUT_DIR / 'kosis_metric_coverage_review.csv'
review_df = pd.read_csv(review_path).fillna('')
labeled = review_df[review_df['manual_label'].isin(['MATCH','PARTIAL','NO_MATCH','NEED_REVIEW'])]
if len(labeled):
    strict_coverage = (labeled['manual_label'] == 'MATCH').mean()
    broad_coverage = labeled['manual_label'].isin(['MATCH','PARTIAL']).mean()
    print(f'엄격한 검증 가능률: {strict_coverage:.1%} / 판단불가 예상: {1-strict_coverage:.1%}')
    print(f'부분 일치 포함 가능률: {broad_coverage:.1%} / 판단불가 예상: {1-broad_coverage:.1%}')
else:
    print('manual_label 검토 후 다시 실행하세요.')

## 6. 결론 및 KOSIS API 활용 원칙

1. **이름 직접 매칭만으로는 부족하다.** 뉴스 표현은 짧고 일상적이며 통계표명은 행정적·구조적이다. 핵심 지표, 대상, 지역, 기간 키워드로 후보를 검색하고 메타데이터로 재검증한다.
2. **상대 시점은 기사 발행일을 기준으로 절대 시점으로 바꾼다.** `올해/지난해`뿐 아니라 월·분기·누계 여부를 함께 추출한다. 최신 공표자료가 기사 기준 시점보다 늦게 발표될 수 있다는 점도 확인한다.
3. **단위와 지표 정의를 분리한다.** 명/천명/만명은 환산할 수 있지만 `%`와 `%p`, 수준값과 증감률, 명목과 실질은 자동 동일시하면 안 된다.
4. **통계 존재율은 검색 hit율이 아니다.** 표 후보의 지표 정의·모집단·지역·기간·단위가 모두 맞아야 `MATCH`다. 사람 검토 표본으로 실제 판단불가율을 추정한다.
5. **API 호출 순서:** 통합검색 → 표 후보 → `TBL/PRD/ITM` 메타데이터 → 분류·항목 선택 → 수치 조회 → 단위/시점 정렬 → 비교 판정 → 근거 설명.
6. **판정 권고:** `일치 / 대체로 일치 / 불일치 / 맥락 누락 / 비교기준 오류 / 공식통계 미발견 / 판단불가`로 운영한다.